# Papers Data Generator

Generates JS data modules for the Papers section visualizations.

**Output files** (in `site/data/papers/`):
- `papersPerYearDataModule.js` → papersPerYear.js
- `papersByConferenceData.js` → papersByConference.js
- `papersByPublicationData.js` → papersByPublication.js
- `citationsHistogramData.js` → citationsHistogram.js
- `awardsData.js` → awardsWaffleIcons.js
- `awardsDetailData.js` → awardsWaffleIcons.js
- `topicsTreemapData.js` → topicsTreemap.js

In [17]:
import pandas as pd
import numpy as np
import json
import re
import ast
from pathlib import Path

# Paths
SRC = Path("../data/processed/dataset_clean.csv")
OUT = Path("../site/data/papers")
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(SRC)
print(f"Loaded: {len(df):,} papers")

Loaded: 3,530 papers


---
## 1. Papers per Year

In [18]:
# Count papers per year
counts = df["Year"].value_counts().sort_index().reset_index()
counts.columns = ["year", "count"]

# Stats
total = int(counts["count"].sum())
avg = int(round(total / len(counts)))
peak_idx = counts["count"].idxmax()
peak_year = int(counts.loc[peak_idx, "year"])
peak_count = int(counts.loc[peak_idx, "count"])

# JS output
data = counts.to_dict(orient="records")
stats = {"total": total, "avgPerYear": avg, "peakYear": peak_year, "peakCount": peak_count}

js = (
    "/** AUTO-GENERATED */\n"
    f"export const papersPerYearData = {json.dumps(data)};\n\n"
    f"export const papersPerYearStats = {json.dumps(stats)};\n"
)
(OUT / "papersPerYearDataModule.js").write_text(js)
print(f"✓ papersPerYearDataModule.js | {total:,} papers, peak: {peak_year}")

✓ papersPerYearDataModule.js | 3,530 papers, peak: 2020


---
## 2. Papers by Conference

In [19]:
# Map conference names
CONF_MAP = {"Vis": "vis", "InfoVis": "infovis", "VAST": "vast", "SciVis": "scivis"}
CONF_KEYS = ["vis", "infovis", "vast", "scivis"]
CONF_LABELS = {"vis": "Vis", "infovis": "InfoVis", "vast": "VAST", "scivis": "SciVis"}

df["conf"] = df["Conference"].map(CONF_MAP)
wide = df.dropna(subset=["conf"]).groupby(["Year", "conf"]).size().unstack(fill_value=0).reset_index()

for k in CONF_KEYS:
    if k not in wide.columns:
        wide[k] = 0

wide = wide[["Year"] + CONF_KEYS].sort_values("Year")

# JS output
records = [{"year": int(r.Year), **{k: int(getattr(r, k)) for k in CONF_KEYS}} for r in wide.itertuples()]

js = (
    "/** AUTO-GENERATED */\n"
    f"export const papersByConferenceData = {json.dumps(records)};\n\n"
    f"export const conferenceKeys = {json.dumps(CONF_KEYS)};\n\n"
    f"export const conferenceLabels = {json.dumps(CONF_LABELS)};\n"
)
(OUT / "papersByConferenceData.js").write_text(js)
print(f"✓ papersByConferenceData.js | {len(records)} years")

✓ papersByConferenceData.js | 35 years


---
## 3. Papers by Publication Type

In [20]:
# Map publication types
PUB_MAP = {"J": "journal", "C": "conference"}
PUB_KEYS = ["conference", "journal"]
PUB_LABELS = {"conference": "Conference", "journal": "Journal (TVCG)"}

df["pub"] = df["PaperType"].map(PUB_MAP)
wide = df.dropna(subset=["pub"]).groupby(["Year", "pub"]).size().unstack(fill_value=0).reset_index()

for k in PUB_KEYS:
    if k not in wide.columns:
        wide[k] = 0

wide = wide[["Year"] + PUB_KEYS].sort_values("Year")

# JS output
records = [{"year": int(r.Year), **{k: int(getattr(r, k)) for k in PUB_KEYS}} for r in wide.itertuples()]

js = (
    "/** AUTO-GENERATED */\n"
    f"export const papersByPublicationData = {json.dumps(records)};\n\n"
    f"export const publicationKeys = {json.dumps(PUB_KEYS)};\n\n"
    f"export const publicationLabels = {json.dumps(PUB_LABELS)};\n"
)
(OUT / "papersByPublicationData.js").write_text(js)
print(f"✓ papersByPublicationData.js | {len(records)} years")

✓ papersByPublicationData.js | 35 years


---
## 4. Citations Histogram

In [21]:
# Merge citation sources: max(CrossRef, Aminer)
c1 = pd.to_numeric(df.get("CitationCount_CrossRef"), errors="coerce")
c2 = pd.to_numeric(df.get("AminerCitationCount"), errors="coerce")

cit = c1.fillna(c2)
both = c1.notna() & c2.notna()
cit.loc[both] = np.maximum(c1[both], c2[both])
cit = cit.fillna(0).clip(lower=0).astype(int)

records = cit.tolist()
vals = np.array(records)

# Stats
stats = {
    "median": int(np.median(vals)),
    "mean": round(float(vals.mean()), 1),
    "max": int(vals.max()),
    "papersWith100Plus": int((vals >= 100).sum()),
    "papersWith500Plus": int((vals >= 500).sum())
}

# Bins
thresholds = [0, 5, 10, 20, 35, 50, 75, 100, 150, 200, 500, 1000, 1500, 4000]
labels = [f"{thresholds[i]}-{thresholds[i+1]}" for i in range(len(thresholds)-1)] + [f"{thresholds[-1]}+"]
bins = {"thresholds": thresholds + [max(4000, vals.max()) + 500], "labels": labels}

# Top cited
df["_cit"] = cit
top2 = df.nlargest(2, "_cit")[["Title", "Year", "_cit"]].rename(columns={"_cit": "Citations"})
top_papers = top2.to_dict(orient="records")

# JS output
js = (
    "/** AUTO-GENERATED */\n"
    f"export const citationsHistogramData = {json.dumps(records)};\n\n"
    f"export const citationStats = {json.dumps(stats)};\n\n"
    f"export const histogramBins = {json.dumps(bins)};\n\n"
    f"export const topCitedPapers = {json.dumps(top_papers)};\n"
)
(OUT / "citationsHistogramData.js").write_text(js)
print(f"✓ citationsHistogramData.js | median: {stats['median']}, max: {stats['max']}")

✓ citationsHistogramData.js | median: 39, max: 3795


---
## 5. Awards Data

In [22]:
# Award config
VALID_AWARDS = {"BP", "HM", "TT", "BA", "BCS"}
AWARD_META = {
    "BP": {"type": "Best Paper", "icon": "🏆"},
    "HM": {"type": "Honorable Mention", "icon": "🎖️"},
    "TT": {"type": "Test of Time", "icon": "⏰"},
    "BA": {"type": "Best Application Paper", "icon": "🚀"},
    "BCS": {"type": "Best Case Study", "icon": "📚"}
}

def parse_awards(x):
    if pd.isna(x): return []
    s = str(x).strip()
    if not s or s.lower() in {"nan", "none"}: return []
    if s.startswith("["):
        try: parts = ast.literal_eval(s)
        except: parts = [s]
    else: parts = [s]
    tokens = []
    for p in parts:
        tokens.extend(re.split(r"[;,|/+\s]+", str(p).upper()))
    return [t for t in tokens if t in VALID_AWARDS]

df["_awards"] = df["Award"].apply(parse_awards)

# Counts
flat = [a for lst in df["_awards"] for a in lst]
counts = pd.Series(flat).value_counts().reindex(["BP", "HM", "TT", "BA", "BCS"]).fillna(0).astype(int)

# Stats
papers_awarded = int((df["_awards"].apply(len) > 0).sum())
stats = {
    "total": int(counts.sum()),
    "papersAwarded": papers_awarded,
    "papersTotal": len(df),
    "percentageAwarded": round(papers_awarded / len(df) * 100, 2)
}

# awardsData
awards_data = [
    {"type": AWARD_META[c]["type"], "count": int(counts[c]), "icon": AWARD_META[c]["icon"], "code": c}
    for c in ["BP", "HM", "TT", "BA", "BCS"]
]

js = (
    "/** AUTO-GENERATED */\n"
    f"export const awardsData = {json.dumps(awards_data, ensure_ascii=False)};\n\n"
    f"export const awardStats = {json.dumps(stats)};\n\n"
    "export const pictogramCellValue = 1;\n"
)
(OUT / "awardsData.js").write_text(js)
print(f"✓ awardsData.js | {stats['papersAwarded']} awarded papers")

✓ awardsData.js | 271 awarded papers


---
## 6. Awards Detail Data

In [23]:
# Check for Graphics Replicability Stamp
def has_stamp(x):
    if pd.isna(x): return False
    return str(x).strip().lower() not in {"", "nan", "none", "0", "false"}

df["_stamp"] = df["GraphicsReplicabilityStamp"].apply(has_stamp) if "GraphicsReplicabilityStamp" in df.columns else False

# Build detail by type
awarded = df[df["_awards"].apply(len) > 0].copy()
awards_by_type = {}

for code in ["BP", "HM", "TT", "BA", "BCS"]:
    papers = []
    for _, r in awarded.iterrows():
        if code in r["_awards"]:
            papers.append({
                "year": int(r["Year"]) if pd.notna(r["Year"]) else None,
                "title": str(r["Title"]),
                "otherAwards": [a for a in r["_awards"] if a != code],
                "otherAwardsLabels": [AWARD_META[a]["type"] for a in r["_awards"] if a != code],
                "hasGraphicsReplicabilityStamp": r["_stamp"]
            })
    papers.sort(key=lambda x: x["year"] or 0, reverse=True)
    awards_by_type[code] = {
        "label": AWARD_META[code]["type"],
        "icon": AWARD_META[code]["icon"],
        "count": len(papers),
        "papers": papers
    }

js = (
    "/** AUTO-GENERATED */\n"
    f"export const awardsDetailByType = {json.dumps(awards_by_type, ensure_ascii=False)};\n"
)
(OUT / "awardsDetailData.js").write_text(js)
print(f"✓ awardsDetailData.js | BP:{awards_by_type['BP']['count']}, HM:{awards_by_type['HM']['count']}")

✓ awardsDetailData.js | BP:83, HM:144


---
## 7. Topics Treemap

In [24]:
# Load topic mapping
topics_src = Path("../data/processed/topic_macro_mapping_renamed.csv")
tdf = pd.read_csv(topics_src)

# Build treemap structure
treemap = {"name": "Topics", "children": []}

for macro, sub in tdf.groupby("macro_name"):
    children = [
        {"name": r.topic_name, "value": int(getattr(r, "count")), "topic_id": int(r.topic_id)}
        for r in sub.sort_values("count", ascending=False).itertuples()
    ]
    treemap["children"].append({"name": macro, "children": children})

# JS output (topic colors moved to centralized palette)
js = (
    "/** AUTO-GENERATED */\n"
    f"export const topicsTreemapData = {json.dumps(treemap, ensure_ascii=False)};\n"
)
(OUT / "topicsTreemapData.js").write_text(js)
print(f"✓ topicsTreemapData.js | {len(treemap['children'])} macro categories")

✓ topicsTreemapData.js | 12 macro categories


---
## Summary

Generated files:
```
site/data/papers/
├── papersPerYearDataModule.js
├── papersByConferenceData.js
├── papersByPublicationData.js
├── citationsHistogramData.js
├── awardsData.js
├── awardsDetailData.js
└── topicsTreemapData.js
```
